# 28b · CXR-LT — tiền xử lý phần ảnh còn thiếu

`28_cxrlt_inventory.ipynb` C6 đã đo: **346,578 / 377,110 ảnh (91.9%) đã có shard**, còn
**30,532 ảnh** chưa. Notebook này xử lý đúng phần thiếu đó.

Đây **không phải tải về**. `04a_preprocess_images.ipynb` C2 cho thấy toàn bộ MIMIC-CXR-JPG đã nằm
sẵn trên Drive (`SOURCE='drive'`), nên việc cần làm chỉ là **tiền xử lý**: resize theo đúng quy tắc
của RAD-DINO về 560px, center-crop, xám hoá, nén JPEG q90, đóng thành `.tar`.

> **Không đụng vào `cxr_560_shards/`.** Shard mới ghi sang thư mục **riêng**
> `cxr_560_shards_cxrlt/` với `_work_list.parquet` riêng. Bộ shard cũ và tính tái lập của
> `cohort_v2` giữ nguyên vẹn; `31_cxrlt_embed.ipynb` sẽ đọc **cả hai** thư mục.

Cổng quan trọng nhất nằm ở C2: **mở thử một ảnh trong shard cũ và kiểm nó đúng `STORE_SIZE` và
đúng số kênh**. Nếu shard cũ dùng chế độ tiền xử lý khác thì trộn hai bộ sẽ cho Stage 3 hai chuẩn
ảnh khác nhau — hỏng âm thầm, không có lỗi nào bắn ra.

**CPU + Drive I/O, không cần GPU.** ~31 shard. Ở tốc độ 04a đo được (49 img/s, trần 83 MB/s của
VM Colab) thì khoảng **10–15 phút**. Chạy được **song song với `29_cxrlt_ehr_base.ipynb`** vì
notebook kia chỉ tốn BigQuery.

## C0 · Config — hằng số giữ **y hệt** `04a`

In [1]:
import os, io, sys, time, tarfile, json
import numpy as np, pandas as pd
from concurrent.futures import ThreadPoolExecutor
from PIL import Image

try:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR = '/content/drive/MyDrive/MIMIC_CXR/planB_out'
except Exception:
    DATA_DIR = './planB_out'

# ---- moi hang so duoi day PHAI khop 04a_preprocess_images.ipynb C2 ----
# Lech mot cai thoi la shard moi khac chuan shard cu, va Stage 3 se doc hai loai anh
# ma khong co gi bao. C2 ben duoi kiem lai bang cach mo anh that trong shard cu.
SOURCE        = 'drive'
DRIVE_IMG     = '/content/drive/MyDrive/MIMIC_CXR/mimic-cxr-jpg-2.1.0.physionet.org/files'
SHARD_SIZE    = 1000
GRAYSCALE     = True
JPEG_Q        = 90
N_THREADS     = 32
CROP_HEADROOM = 1.08

# ---- vao / ra ----
OLD_SHARD_DIR = os.path.join(DATA_DIR, 'cxr_560_shards')          # CHI DOC, khong ghi
OUT_DIR       = os.path.join(DATA_DIR, 'cxrlt_2024')
SHARD_DIR     = os.path.join(DATA_DIR, 'cxr_560_shards_cxrlt')    # shard moi ghi o day
os.makedirs(SHARD_DIR, exist_ok=True)

# ---- chia cong viec cho nhieu runtime (tuy chon) ----
# Tran ~80 MB/s la PER VM, nen runtime thu hai co tran rieng. Cac khoang phai roi nhau;
# shard dung rename-on-complete + skip-if-exists nen chong nhau chi phi pham, khong hong.
SHARD_FROM = None      # None -> 0
SHARD_TO   = None      # None -> N_SHARDS

P_IDX = os.path.join(OUT_DIR, 'cxrlt_index.parquet')
assert os.path.exists(P_IDX), f'khong thay {P_IDX} -> chay 28_cxrlt_inventory.ipynb truoc'
assert os.path.isdir(DRIVE_IMG), (
    f'khong thay anh goc: {DRIVE_IMG}\n'
    '  -> day la ban MIMIC-CXR-JPG tren Drive ma 04a dung. Kiem lai duong dan.')
print(f'anh goc   : {DRIVE_IMG}')
print(f'shard cu  : {OLD_SHARD_DIR}  (CHI DOC)')
print(f'shard moi : {SHARD_DIR}')

Mounted at /content/drive
anh goc   : /content/drive/MyDrive/MIMIC_CXR/mimic-cxr-jpg-2.1.0.physionet.org/files
shard cu  : /content/drive/MyDrive/MIMIC_CXR/planB_out/cxr_560_shards  (CHI DOC)
shard moi : /content/drive/MyDrive/MIMIC_CXR/planB_out/cxr_560_shards_cxrlt


## C1 · `STORE_SIZE` lấy từ chính processor của RAD-DINO

In [2]:
from transformers import AutoImageProcessor
proc = AutoImageProcessor.from_pretrained('microsoft/rad-dino')

MODEL_SIZE = (proc.crop_size['height'] if getattr(proc, 'do_center_crop', False)
              else (proc.size.get('shortest_edge') or proc.size.get('height')))
STORE_SIZE = int(round(MODEL_SIZE * CROP_HEADROOM))
SHORTEST_EDGE = bool(getattr(proc, 'do_center_crop', False)) or \
                'shortest_edge' in (proc.size or {})

print(f'MODEL_SIZE (ViT nhin thay) : {MODEL_SIZE}')
print(f'STORE_SIZE (luu tren dia)  : {STORE_SIZE}   (+{CROP_HEADROOM:.0%} headroom cho crop)')
print(f'quy tac resize: {"canh ngan -> center crop (giu ty le)" if SHORTEST_EDGE else "ep vuong"}')

preprocessor_config.json:   0%|          | 0.00/756 [00:00<?, ?B/s]

MODEL_SIZE (ViT nhin thay) : 518
STORE_SIZE (luu tren dia)  : 559   (+108% headroom cho crop)
quy tac resize: canh ngan -> center crop (giu ty le)


## C2 · Cổng chặn — shard cũ có **cùng chế độ** không?

Mở một ảnh thật trong `cxr_560_shards/` và kiểm kích thước + số kênh. Đây là thứ duy nhất chứng
minh được rằng shard mới sẽ **ghép liền** với shard cũ chứ không phải hai chuẩn khác nhau.

In [3]:
_tars = sorted(f for f in os.listdir(OLD_SHARD_DIR) if f.endswith('.tar'))
assert _tars, f'khong thay .tar nao trong {OLD_SHARD_DIR}'
with tarfile.open(os.path.join(OLD_SHARD_DIR, _tars[0])) as _t:
    _m = [m for m in _t.getmembers() if m.name.endswith('.jpg')][:3]
    assert _m, 'shard cu khong co member .jpg nao'
    for _mm in _m:
        _im = Image.open(io.BytesIO(_t.extractfile(_mm).read()))
        print(f'  {_mm.name[:20]}...  size={_im.size}  mode={_im.mode}  {_mm.size/1024:.0f} KB')
        assert _im.size == (STORE_SIZE, STORE_SIZE), (
            f'shard cu luu {_im.size} nhung C1 tinh ra STORE_SIZE={STORE_SIZE}.\n'
            '  -> KHONG duoc tron hai bo. Hoac sua CROP_HEADROOM cho khop shard cu,\n'
            '     hoac dung shard moi RIENG cho toan bo CXR-LT (dung lai 04a tu dau).')
        assert _im.mode == ('L' if GRAYSCALE else 'RGB'), \
            f'shard cu mode={_im.mode}, cau hinh nay GRAYSCALE={GRAYSCALE}'
print(f'\nOK  shard cu dung {STORE_SIZE}px, mode={"L" if GRAYSCALE else "RGB"} '
      f'-> shard moi se ghep lien duoc')
print(f'    ({len(_tars)} tar trong bo cu)')

  02aa804e-bde0afdd-11...  size=(559, 559)  mode=L  47 KB
  174413ec-4ec4c1f7-34...  size=(559, 559)  mode=L  39 KB
  2a2277a9-b0ded155-c0...  size=(559, 559)  mode=L  48 KB

OK  shard cu dung 559px, mode=L -> shard moi se ghep lien duoc
    (347 tar trong bo cu)


## C3 · Danh sách ảnh cần xử lý

In [4]:
IDX = pd.read_parquet(P_IDX, columns=['dicom_id', 'study_id', 'subject_id',
                                      'split', 'view_group', 'has_shard'])
work = (IDX[~IDX.has_shard]
          .drop(columns='has_shard')
          .sort_values(['subject_id', 'study_id', 'dicom_id'])
          .reset_index(drop=True))
print(f'CXR-LT           : {len(IDX):,} anh')
print(f'da co shard cu   : {int(IDX.has_shard.sum()):,}')
print(f'CAN XU LY o day  : {len(work):,}')

# --- doi chieu voi _work_list cu: nhung anh nay that su chua co ---
_wl = os.path.join(OLD_SHARD_DIR, '_work_list.parquet')
if os.path.exists(_wl):
    _old = set(pd.read_parquet(_wl, columns=['dicom_id']).dicom_id)
    _dup = set(work.dicom_id) & _old
    assert not _dup, f'{len(_dup)} anh da co trong bo cu ma van bi xep vao viec -> has_shard sai'
    print(f'doi chieu _work_list cu: khong trung mot anh nao ({len(_old):,} anh trong bo cu)')

assert len(work) > 0, 'khong con anh nao thieu -> khong can chay notebook nay'
print(f'\ntheo split : {work.split.value_counts().to_dict()}')
print(f'theo view  : {work.view_group.value_counts().to_dict()}')

work['shard'] = np.arange(len(work)) // SHARD_SIZE
N_SHARDS = int(work.shard.max()) + 1
print(f'\n-> {N_SHARDS} shard x toi da {SHARD_SIZE} anh')
print(f'   uoc tinh: {len(work)/49/60:.0f}-{len(work)/30/60:.0f} phut '
      f'(04a do duoc 49 img/s tren Drive @32 thread)')

# --- kiem duong dan anh goc co that su phan giai duoc khong ---
_r = work.iloc[0]
_s = str(int(_r.subject_id))
_rel = f'p{_s[:2]}/p{_s}/s{int(_r.study_id)}/{_r.dicom_id}.jpg'
_p = os.path.join(DRIVE_IMG, _rel)
assert os.path.exists(_p), (
    f'anh goc dau tien khong ton tai:\n  {_p}\n'
    '  -> kiem DRIVE_IMG, hoac ban MIMIC-CXR-JPG tren Drive thieu file nay.')
print(f'\nmau duong dan OK: .../{_rel}  ({os.path.getsize(_p)/1e6:.2f} MB)')

CXR-LT           : 377,110 anh
da co shard cu   : 346,578
CAN XU LY o day  : 30,532
doi chieu _work_list cu: khong trung mot anh nao (346,578 anh trong bo cu)

theo split : {'train': 20663, 'test': 6588, 'val': 3281}
theo view  : {'frontal': 12638, 'unknown': 11743, 'lateral': 6151}

-> 31 shard x toi da 1000 anh
   uoc tinh: 10-17 phut (04a do duoc 49 img/s tren Drive @32 thread)

mau duong dan OK: .../p10/p10001122/s53447138/8039752c-2ea661b7-16f1eafe-055b7e7b-dbd4cdd1.jpg  (1.11 MB)


## C4 · Dựng shard

Port nguyên `04a_preprocess_images.ipynb` C3 — cùng `_prep`, cùng `_read_bytes`, cùng
rename-on-complete và skip-if-exists (chạy lại là tiếp tục được, không làm lại từ đầu).

In [5]:
def _read_bytes(sub, st, dic):
    s = str(int(sub)); rel = f'p{s[:2]}/p{s}/s{int(st)}/{dic}.jpg'
    with open(os.path.join(DRIVE_IMG, rel), 'rb') as f:
        return f.read()

def _prep(raw):
    '''Tai lap dung quy tac resize cua RAD-DINO, o STORE_SIZE.'''
    im = Image.open(io.BytesIO(raw))
    im = im.convert('L') if GRAYSCALE else im.convert('RGB')
    if SHORTEST_EDGE:
        w, h = im.size; s = STORE_SIZE / min(w, h)
        im = im.resize((max(1, round(w*s)), max(1, round(h*s))), Image.BICUBIC)
        w, h = im.size; l, t = (w-STORE_SIZE)//2, (h-STORE_SIZE)//2
        im = im.crop((l, t, l+STORE_SIZE, t+STORE_SIZE))
    else:
        im = im.resize((STORE_SIZE, STORE_SIZE), Image.BICUBIC)
    buf = io.BytesIO(); im.save(buf, 'JPEG', quality=JPEG_Q, optimize=True)
    return buf.getvalue()

def _one(r):
    try:
        return r.dicom_id, _prep(_read_bytes(r.subject_id, r.study_id, r.dicom_id)), None
    except Exception as e:
        return r.dicom_id, None, f'{type(e).__name__}: {e}'

def build_shard(i):
    final = os.path.join(SHARD_DIR, f'cxrlt_{i:05d}.tar')
    if os.path.exists(final): return 'skip', 0, []
    tmp  = final + '.tmp'                     # rename-on-complete: lan chay bi giet
    rows = work[work.shard == i]              # khong bao gio de lai mot tar trong ma nhin nhu that
    errs = []
    with tarfile.open(tmp, 'w') as tar, ThreadPoolExecutor(N_THREADS) as ex:
        for dic, blob, err in ex.map(_one, [r for r in rows.itertuples()]):
            if err: errs.append((dic, err)); continue
            info = tarfile.TarInfo(f'{dic}.jpg'); info.size = len(blob)
            tar.addfile(info, io.BytesIO(blob))
    os.replace(tmp, final)
    return 'done', len(rows) - len(errs), errs

LO = 0 if SHARD_FROM is None else int(SHARD_FROM)
HI = N_SHARDS if SHARD_TO is None else int(SHARD_TO)
assert 0 <= LO < HI <= N_SHARDS, f'khoang [{LO},{HI}) khong hop le voi {N_SHARDS} shard'
_have = len([f for f in os.listdir(SHARD_DIR) if f.endswith('.tar')])
print(f'runtime nay dung shard [{LO}, {HI}) / {N_SHARDS}   ({_have} da co tren dia)')

t0, n_ok, all_errs, n_built = time.time(), 0, [], 0
for i in range(LO, HI):
    st, n, errs = build_shard(i)
    n_ok += n; all_errs += errs
    if st == 'done':
        n_built += 1
        el   = time.time() - t0
        rate = n_ok / max(el, 1e-9)
        left = (HI - i - 1) * SHARD_SIZE / max(rate, 1e-9)
        print(f'  shard {i+1:>4}/{HI}  {rate:5.1f} img/s  elapsed {el/60:5.1f}m  '
              f'eta {left/60:5.1f}m  loi {len(all_errs)}', end='\r')
print(f'\n\nxong. shard dung o runtime nay: {n_built} | anh ghi duoc: {n_ok:,} | '
      f'loi: {len(all_errs)}')
if all_errs:
    pd.DataFrame(all_errs, columns=['dicom_id', 'error']).to_csv(
        os.path.join(SHARD_DIR, '_errors.csv'), index=False)
    print('vai loi dau:', all_errs[:5])

runtime nay dung shard [0, 31) / 31   (0 da co tren dia)
  shard   31/31   24.0 img/s  elapsed  21.2m  eta   0.0m  loi 0

xong. shard dung o runtime nay: 31 | anh ghi duoc: 30,532 | loi: 0


## C5 · Kiểm chứng + ghi `_work_list.parquet` cho bộ mới

In [6]:
tars = sorted(f for f in os.listdir(SHARD_DIR) if f.endswith('.tar'))
missing_sh = [i for i in range(N_SHARDS)
              if not os.path.exists(os.path.join(SHARD_DIR, f'cxrlt_{i:05d}.tar'))]
if missing_sh:
    print(f'!! thieu {len(missing_sh)} shard, vd {missing_sh[:8]}')
    print(f'   -> dat SHARD_FROM/SHARD_TO vao khoang do roi chay lai C4')
assert not missing_sh, f'thieu {len(missing_sh)} shard'

# --- doc THAT ten member trong tung tar: bang chung anh nam trong do, khong phai suy dien ---
got = set()
for k, f in enumerate(tars, 1):
    with tarfile.open(os.path.join(SHARD_DIR, f)) as t:
        got |= {n[:-4] for n in t.getnames() if n.endswith('.jpg')}
    print(f'  doc {k}/{len(tars)} tar', end='\r')
tot = sum(os.path.getsize(os.path.join(SHARD_DIR, f)) for f in tars)
print(f'\n{len(tars)} tar | {len(got):,} anh | {tot/1e9:.2f} GB')

need = set(work.dicom_id)
miss = need - got
if miss:
    print(f'\n!! {len(miss):,} anh KHONG vao duoc shard (xem _errors.csv)')
    print(f'   vd: {sorted(miss)[:5]}')
assert not miss, f'{len(miss)} anh thieu -> chua du de chay 31_cxrlt_embed.ipynb'
assert not (got - need), f'{len((got-need)):,} anh la lot vao shard moi'
print('moi anh can xu ly deu co mat trong shard, khong du thua')

# --- kich thuoc anh moi phai bang shard cu ---
with tarfile.open(os.path.join(SHARD_DIR, tars[0])) as t:
    _m = next(m for m in t.getmembers() if m.name.endswith('.jpg'))
    _im = Image.open(io.BytesIO(t.extractfile(_m).read()))
    assert _im.size == (STORE_SIZE, STORE_SIZE) and _im.mode == ('L' if GRAYSCALE else 'RGB')
print(f'anh moi: {_im.size} mode={_im.mode} -> khop shard cu (C2 da kiem)')

WL = work[['dicom_id', 'study_id', 'subject_id', 'view_group', 'split', 'shard']]
WL.to_parquet(os.path.join(SHARD_DIR, '_work_list.parquet'), index=False)
json.dump(dict(store_size=STORE_SIZE, model_size=MODEL_SIZE, grayscale=GRAYSCALE,
               jpeg_q=JPEG_Q, crop_headroom=CROP_HEADROOM, shortest_edge=SHORTEST_EDGE,
               shard_size=SHARD_SIZE, n_shards=N_SHARDS, n_images=len(work),
               tar_prefix='cxrlt_', nguon=DRIVE_IMG,
               ghi_chu='bo sung cho cxr_560_shards; cung che do tien xu ly'),
          open(os.path.join(SHARD_DIR, '_shard_meta.json'), 'w'), indent=1)
print(f'ghi _work_list.parquet + _shard_meta.json vao {SHARD_DIR}')

_old_n = len(pd.read_parquet(os.path.join(OLD_SHARD_DIR, '_work_list.parquet'),
                             columns=['dicom_id'])) if os.path.exists(
                                 os.path.join(OLD_SHARD_DIR, '_work_list.parquet')) else 0
print(f'''
================================================================
DO PHU ANH CUA CXR-LT

  bo cu  cxr_560_shards       : {_old_n:,} anh   (khong bi dung toi)
  bo moi cxr_560_shards_cxrlt : {len(got):,} anh
  ------------------------------------------------------------
  tong                        : {_old_n + len(got):,} / {len(IDX):,}

31_cxrlt_embed.ipynb phai doc CA HAI thu muc.
================================================================''')

  doc 31/31 tar
31 tar | 30,532 anh | 1.49 GB
moi anh can xu ly deu co mat trong shard, khong du thua
anh moi: (559, 559) mode=L -> khop shard cu (C2 da kiem)
ghi _work_list.parquet + _shard_meta.json vao /content/drive/MyDrive/MIMIC_CXR/planB_out/cxr_560_shards_cxrlt

DO PHU ANH CUA CXR-LT

  bo cu  cxr_560_shards       : 346,578 anh   (khong bi dung toi)
  bo moi cxr_560_shards_cxrlt : 30,532 anh
  ------------------------------------------------------------
  tong                        : 377,110 / 377,110

31_cxrlt_embed.ipynb phai doc CA HAI thu muc.
